# Tokenization

Tokenization is the process of taking text (such as a sentence) and breaking it into individual terms (usually words). A simple Tokenizer class provides this functionality. The example below shows how to split sentences into sequences of words.

RegexTokenizer allows more advanced tokenization based on regular expression (regex) matching. By default, the parameter “pattern” (regex, default: "\\s+") is used as delimiters to split the input text. Alternatively, users can set parameter “gaps” to false indicating the regex “pattern” denotes “tokens” rather than splitting gaps, and find all matching occurrences as the tokenization result.

Commonly used regular expression symbols:

|symbol| meaning|
|---|------------------------|
|^ |means "begin with"|
|$ |means "ends with"|
|. 	|means any one character |
|\w |	means \[a-zA-Z0-9_\]|
|\W | a non-alphanumeric |
|\d |	means digital numbers|
|\D | any non-digit|
|\t |	means a tab |
|\s | \[ \r\t\n\f\]  whitespace (space, tab)|
|\S |non-whitespace|
|\+ | means 1 or more| 
|\s+| means 1 or more whitespaces|


## Create PySpark environment and import library

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master('local').appName('MyTokenization').getOrCreate()

from pyspark.ml.feature import Tokenizer, RegexTokenizer
from pyspark.sql.functions import col, udf
from pyspark.sql.types import IntegerType

## Create df

In [3]:
sen_df = spark.createDataFrame([
    (0, 'Hi I heard about Spark.'),
    (1, 'I wish java could use case classes.'),
    (2, 'Logistic,regression,model,are,neat.')
], ['id', 'sentence'])

sen_df.show()

+---+--------------------+
| id|            sentence|
+---+--------------------+
|  0|Hi I heard about ...|
|  1|I wish java could...|
|  2|Logistic,regressi...|
+---+--------------------+



## Tokenize

In [4]:
tokenizer = Tokenizer(inputCol = 'sentence', outputCol = 'words')
tokenized = tokenizer.transform(sen_df)
tokenized.show(truncate=False)

+---+-----------------------------------+-------------------------------------------+
|id |sentence                           |words                                      |
+---+-----------------------------------+-------------------------------------------+
|0  |Hi I heard about Spark.            |[hi, i, heard, about, spark.]              |
|1  |I wish java could use case classes.|[i, wish, java, could, use, case, classes.]|
|2  |Logistic,regression,model,are,neat.|[logistic,regression,model,are,neat.]      |
+---+-----------------------------------+-------------------------------------------+



## Tokenize with token count

In [5]:
count_tokens = udf(lambda words: len(words), IntegerType())
temp = tokenized.withColumn('tokens', count_tokens(col('words')))
temp.show()

+---+--------------------+--------------------+------+
| id|            sentence|               words|tokens|
+---+--------------------+--------------------+------+
|  0|Hi I heard about ...|[hi, i, heard, ab...|     5|
|  1|I wish java could...|[i, wish, java, c...|     7|
|  2|Logistic,regressi...|[logistic,regress...|     1|
+---+--------------------+--------------------+------+



## Process the tokens with regex

In [6]:
# pattern = '\W' means non-alphanumeric characters
regex_tokenizer = RegexTokenizer(inputCol = 'sentence', outputCol = 'words', pattern = '\W')
regex_tokenized = regex_tokenizer.transform(sen_df)
regex_tokenized.show()

+---+--------------------+--------------------+
| id|            sentence|               words|
+---+--------------------+--------------------+
|  0|Hi I heard about ...|[hi, i, heard, ab...|
|  1|I wish java could...|[i, wish, java, c...|
|  2|Logistic,regressi...|[logistic, regres...|
+---+--------------------+--------------------+



In [7]:
regex_tokenized.withColumn('tokens', count_tokens(col('words'))).show()

+---+--------------------+--------------------+------+
| id|            sentence|               words|tokens|
+---+--------------------+--------------------+------+
|  0|Hi I heard about ...|[hi, i, heard, ab...|     5|
|  1|I wish java could...|[i, wish, java, c...|     7|
|  2|Logistic,regressi...|[logistic, regres...|     5|
+---+--------------------+--------------------+------+



## Process text file input

In [10]:
df_text = spark.read.text("data/2.txt")
df_text.printSchema()
df_text.show()

root
 |-- value: string (nullable = true)

+--------------------+
|               value|
+--------------------+
|Lorem Ipsum is si...|
|The standard chun...|
|There are many va...|
|Lorem ipsum dolor...|
|Mauris ut dapibus...|
|Lorem Ipsum is si...|
|Contrary to popul...|
|The standard chun...|
+--------------------+



In [12]:
#tokenizer = Tokenizer(inputCol = 'value', outputCol = 'words')
regex_tokenizer = RegexTokenizer(inputCol = 'value', outputCol = 'words', pattern = '\W')
#count_tokens = udf(lambda words: len(words), IntegerType())
tokenized = regex_tokenizer.transform(df_text)
tokenized.withColumn('#_tokens', count_tokens(col('words'))).show(truncate=False)



+-----------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------+--------+
|value                                                                                                                  |words                                                                                                                                    |#_tokens|
+-----------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------+--------+
|Lorem Ipsum is simply dummy text of the printing                                                                       |[lorem, ipsum, is, simply, dummy, text, of, the, printing]              

In [13]:
mydf = spark.createDataFrame([
    (0, "Finland's capital is Helsinki."),
    (1, 'Hewlett-Packard is a big company.'),
    (2, 'Bert is a state-of-art model for AI')
], ['id', 'sentence'])

mydf.show(truncate=False)

+---+-----------------------------------+
|id |sentence                           |
+---+-----------------------------------+
|0  |Finland's capital is Helsinki.     |
|1  |Hewlett-Packard is a big company.  |
|2  |Bert is a state-of-art model for AI|
+---+-----------------------------------+



In [14]:
import re
def removeSomePunctuation(text):
    #return re.sub(r'[^a-z0-9\s]','', text.lower().strip())
    return re.sub(r"[^a-z0-9-'\s]",'', text.lower().strip()) # only keep characters: a-z, 0-9 , - ,and ' in a line

rdd = mydf.rdd.map(lambda x: (x[0], removeSomePunctuation(x[1])))
rdd.collect()
df = rdd.toDF(['id', 'sentence'])
df.show(truncate=False)

+---+-----------------------------------+
|id |sentence                           |
+---+-----------------------------------+
|0  |finland's capital is helsinki      |
|1  |hewlett-packard is a big company   |
|2  |bert is a state-of-art model for ai|
+---+-----------------------------------+



In [15]:
tokenizer = Tokenizer(inputCol = 'sentence', outputCol = 'words')
tokenized = tokenizer.transform(df)
#select a column 
tokenized.select('words').show(truncate=False)

+-------------------------------------------+
|words                                      |
+-------------------------------------------+
|[finland's, capital, is, helsinki]         |
|[hewlett-packard, is, a, big, company]     |
|[bert, is, a, state-of-art, model, for, ai]|
+-------------------------------------------+



In [16]:
tokenized.show(truncate=False)

+---+-----------------------------------+-------------------------------------------+
|id |sentence                           |words                                      |
+---+-----------------------------------+-------------------------------------------+
|0  |finland's capital is helsinki      |[finland's, capital, is, helsinki]         |
|1  |hewlett-packard is a big company   |[hewlett-packard, is, a, big, company]     |
|2  |bert is a state-of-art model for ai|[bert, is, a, state-of-art, model, for, ai]|
+---+-----------------------------------+-------------------------------------------+



In [17]:
s = tokenized.select('words')
print(type(s))
for i in s.rdd.collect():
    print(i)

<class 'pyspark.sql.dataframe.DataFrame'>
Row(words=["finland's", 'capital', 'is', 'helsinki'])
Row(words=['hewlett-packard', 'is', 'a', 'big', 'company'])
Row(words=['bert', 'is', 'a', 'state-of-art', 'model', 'for', 'ai'])


In [18]:
spark.stop()